In [1]:
# ============================================
# 1. Setup
# ============================================

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================
# 1. Mount Dri

# ============================================
# 2. Imports
# ============================================
import os
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

INPUT_DIR = "/content/drive/MyDrive/mast2_dataset"
OUTPUT_DIR = "/content/drive/MyDrive/new_mast2_dataset"

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================
# 3. Crop + Save
# ============================================

def crop_and_save(folder_path, crop_box):
    out_folder = Path(OUTPUT_DIR) / folder_path.name
    out_folder.mkdir(parents=True, exist_ok=True)

    for img_path in folder_path.glob("*.jpg"):
        img = Image.open(img_path)
        cropped = img.crop(crop_box)
        cropped.save(out_folder / img_path.name, quality=95)


# ============================================
# 4. Interactive Slider UI
# ============================================

mast_folders = sorted([d for d in Path(INPUT_DIR).iterdir() if d.is_dir()])
index = 0

def process_next(_=None):
    global index

    clear_output(wait=True)

    if index >= len(mast_folders):
        print("🎉 DONE ALL MASTS")
        return

    mast = mast_folders[index]
    print(f"[{index+1}/{len(mast_folders)}] {mast.name}")

    ref_img = mast / "original.jpg"
    if not ref_img.exists():
        ref_img = list(mast.glob("*.jpg"))[0]

    img = Image.open(ref_img)
    w, h = img.size

    # sliders
    x1 = widgets.IntSlider(0, 0, w, description='x1')
    y1 = widgets.IntSlider(0, 0, h, description='y1')
    x2 = widgets.IntSlider(w, 0, w, description='x2')
    y2 = widgets.IntSlider(h, 0, h, description='y2')

    button = widgets.Button(description="Crop & Next", button_style='success')

    def update_preview(*args):
        plt.figure(figsize=(5,5))
        plt.imshow(img)
        plt.gca().add_patch(
            plt.Rectangle((x1.value, y1.value),
                          x2.value-x1.value,
                          y2.value-y1.value,
                          edgecolor='red',
                          fill=False,
                          linewidth=2)
        )
        plt.axis('off')
        plt.show()

    x1.observe(update_preview, names='value')
    y1.observe(update_preview, names='value')
    x2.observe(update_preview, names='value')
    y2.observe(update_preview, names='value')

    update_preview()

    def on_click(b):
        global index

        crop_box = (x1.value, y1.value, x2.value, y2.value)
        crop_and_save(mast, crop_box)

        print(f"✅ Cropped {mast.name}")

        index += 1
        process_next()

    button.on_click(on_click)

    display(x1, y1, x2, y2, button)


# ============================================
# 5. Start
# ============================================

process_next()

🎉 DONE ALL MASTS
